In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import gc 
gc.collect()

47

In [3]:
import pandas as pd
import numpy as np
import getpass
import sys
import datetime
import io
usr_name = getpass.getuser()
sys.path.append(f'/home/{usr_name}/notebooks/utils')
from spark_utils import *
#import datadicts as dd
from bpm_features import calc_all_features

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [4]:
from subprocess import Popen, PIPE, run
from getpass import getpass

def kinit(username: str):
    """
    Obtain and cache an initial ticket-granting ticket for principal.

    :param username: principal username.
    :return:
    """
    kinit_path = '/usr/bin/kinit'
    kinit_args = [kinit_path, '%s' % username]
    pipe = Popen(kinit_args, stdin=PIPE, stdout=PIPE, stderr=PIPE)
    pipe.communicate(input='{}\n'.format(getpass()).encode('utf-8'))
    klist_result = run(['klist'], stdout=PIPE).stdout.decode('utf-8')
    print(klist_result)
    
kinit("21417984_omega-sbrf-ru@DF.SBRF.RU")

 ················


Ticket cache: FILE:/tmp/krb5cc_1623419065
Default principal: 21417984_omega-sbrf-ru@DF.SBRF.RU

Valid starting       Expires              Service principal
03/19/2026 12:05:12  03/20/2026 11:45:47  krbtgt/DF.SBRF.RU@DF.SBRF.RU
	renew until 03/26/2026 12:05:12



In [5]:
#spark
from tsu_spark_utils import *

spark = get_spark_context_scoring('platon_features_multiclass')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 12:05:20 WARN Client: Exception encountered while connecting to the server 
org.apache.hadoop.ipc.RemoteException(org.apache.hadoop.ipc.StandbyException): Operation category READ is not supported in state standby. Visit https://s.apache.org/sbnn-error
	at org.apache.hadoop.security.SaslRpcClient.saslConnect(SaslRpcClient.java:376)
	at org.apache.hadoop.ipc.Client$Connection.setupSaslConnection(Client.java:623)
	at org.apache.hadoop.ipc.Client$Connection.access$2300(Client.java:414)
	at org.apache.hadoop.ipc.Client$Connection$2.run(Client.java:832)
	at org.apache.hadoop.ipc.Client$Connection$2.run(Client.java:828)
	at java.security.AccessController.doPrivileged(Native Method)
	at javax.security.auth.Subject.doAs(Subject.java:422)
	at org.apache.hadoop.security.UserGroupInformation.doAs(UserGroupInformation.java:1878)
	at org.apache.hadoop.ipc.Client$

In [6]:
spark

# Train

In [ ]:
target = spark.sql('''
select epk_id, last_day(date(report_dt) - interval 13 month) as report_dt, target
from arnsdpsbx_team_ss.bpm_premier_multiclass_new_target_v3_new
''')

target.createOrReplaceTempView('target')
target.count()
target.show()

In [8]:
tables_dict = {'agg':'prx_bpm_client_aggr_custom_rozn_client_aggr.ft_client_aggr_mnth',
               'feedbacks':'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_feedbacks',
               'vsp_visits':'prx_bpm_visiting_vsp_custom_rozn_sscxdata_cxdm.cxdm_visiting_vsp_v2',
               'card_transactions': 'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_card_transactions',
               'e_cod':'prx_bpm_cod_platform_cod.cod_deposit_deposit',
               'idoc': 'prx_bpm_arrests_internal_aiv_deposit.idoc',
               'idoc_acc': 'prx_bpm_arrests_internal_aiv_deposit.idoc_acc',
               'pos_embeddings_fl': 'prx_bpm_pos_emb_custom_rozn_ml360.u_fl_transaction_embeddings' ,
               'embeddings_fl': 'prx_bpm_multimodel_emb_custom_fin_palm_ml.cmn_multimodal_emb_ind_fct',
               'card_transactions_10d': 'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_card_transactions_10d'}


In [9]:
model_name = 'bpm_premier_multiclass_v3_new'
output_scheme = 'arnsdpsbx_team_ss' 
mode = 'append'

In [11]:
calc_all_features(spark, target, tables_dict, model_name, output_scheme, mode)

26/03/18 15:12:29 WARN TaskSetManager: Lost task 3021.0 in stage 11.0 (TID 3090) (pklas-nimb00087.labiac.df.sbrf.ru executor 30): TaskKilled (Stage cancelled)
26/03/18 15:40:13 WARN TaskSetManager: Lost task 2448.0 in stage 11.0 (TID 3296) (pklos-nimb00554.labiac.df.sbrf.ru executor 4): TaskKilled (Stage cancelled)


agg_features: done


arrests_features: done


feedbacks_features: done


card_transactions_features: done


...

aggr_dynamic_features_3: done


26/03/19 01:10:25 WARN DAGScheduler: Broadcasting large task binary with size 5.3 MiB
26/03/19 01:14:18 WARN DAGScheduler: Broadcasting large task binary with size 1419.5 KiB


all_features_train: done
arnsdpsbx_team_ss.bpm_premier_multiclass_v3_new_all_features_train


100%|██████████| 133/133 [12:20<00:00,  5.56s/it]
26/03/19 01:46:49 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB


all_features_fix: done
arnsdpsbx_team_ss.bpm_premier_multiclass_v3_new_all_features_fix_types_train: done


In [7]:
spark.sql('''
select count(epk_id), count(distinct epk_id) from arnsdpsbx_team_ss.bpm_premier_multiclass_v3_new_all_features_fix_types_train
''').show()

+-------------+----------------------+
|count(epk_id)|count(DISTINCT epk_id)|
+-------------+----------------------+
|       800000|                800000|
+-------------+----------------------+



In [8]:
spark.sql('''
select * from arnsdpsbx_team_ss.bpm_premier_multiclass_v3_new_all_features_fix_types_train
''').write.parquet('hdfs://arnsdpsbx/user/team/team_ss/bpm_premier_multiclass_v3_new_all_features_fix_types_train_for_model', mode='overwrite')

26/03/19 08:44:34 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/19 08:44:35 WARN DAGScheduler: Broadcasting large task binary with size 1717.1 KiB


# OOT

In [7]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pyspark.sql.types as T
from tqdm import tqdm
def fix_spark_types(df):

        for col, dtype in tqdm(df.dtypes):
                if "decimal" in dtype:
                    df = df.withColumn(col, F.col(col).cast(T.DoubleType()))

        date_columns = [i for i in df.columns if '_dt' in i]
        for col in tqdm(date_columns):
            df = df.withColumn(col, F.when(F.col(col) > F.to_date(F.lit(pd.Timestamp.max)), F.to_date(F.lit(pd.Timestamp.max))).otherwise(F.col(col))) 

        return df

In [8]:
spark.sql('''
select distinct report_dt from arnsdpsbx_team_ss.bpm_premier_multiclass_new_target_v3_oot
''').show()

+----------+
| report_dt|
+----------+
|2025-11-30|
+----------+



In [9]:
spark.sql('''
select last_day(date('2025-11-30') - interval 13 month)
''').show()

+------------------------------------------+
|last_day(2025-11-30 - INTERVAL '13' MONTH)|
+------------------------------------------+
|                                2024-10-31|
+------------------------------------------+



In [20]:
from catboost import CatBoostClassifier
report_dt = '2024-10-31'

tables_dict = {'agg':'prx_bpm_client_aggr_custom_rozn_client_aggr.ft_client_aggr_mnth',
               #'feedbacks':'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_feedbacks',
               #'vsp_visits':'prx_bpm_visiting_vsp_custom_rozn_sscxdata_cxdm.cxdm_visiting_vsp_v2',
               #'card_transactions': 'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_card_transactions',
               #'e_cod':'prx_bpm_cod_platform_cod.cod_deposit_deposit',
               #'idoc': 'prx_bpm_arrests_internal_aiv_deposit.idoc',
               #'idoc_acc': 'prx_bpm_arrests_internal_aiv_deposit.idoc_acc',
               #'pos_embeddings_fl': 'prx_bpm_pos_emb_custom_rozn_ml360.u_fl_transaction_embeddings' ,
               #'embeddings_fl': 'prx_bpm_multimodel_emb_custom_fin_palm_ml.cmn_multimodal_emb_ind_fct',
              #'card_transactions_10d': 'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_card_transactions_10d'
              }

model_name = 'bpm_premier_multiclass_v4_oot_new'
output_scheme = 'arnsdpsbx_team_ss' 

model = CatBoostClassifier()
model.load_model('final_model_v4_new.cbm')
model_features_name = model.feature_names_

full_agg_columns = spark.read.table(tables_dict['agg']).columns
#full_emb_columns = spark.read.table(tables_dict['pos_embeddings_fl']).columns
#full_ct_columns = [i.lower() for i in spark.read.table(tables_dict['card_transactions']).columns]

features = str(model_features_name).replace('[','').replace(']','').replace("'","").replace('"','')
agg_features = str(set(model_features_name).intersection(set(full_agg_columns))).replace('{','').replace('}','').replace("'","").replace('"','')
#pos_emb_features = list(set(model_features_name).intersection(set(full_emb_columns)))
#card_transactions_features = str(set(model_features_name).intersection(set(full_ct_columns))).replace('{','').replace('}','').replace("'","").replace('"','')

In [21]:
aggr_dp_columns = []
for col in full_agg_columns:
    if (("crd" in col[0:3]) or ("dep" in col[0:3]) or ("srv" in col[0:3])):
        if (('3'not in col) 
             and ('6'not in col) 
             and ('9'not in col) 
             and ('12'not in col) 
             and ('dt'not in col) 
             and ('id'not in col) 
             and ('nflag'not in col)
             and ('cd'not in col)
             and ('dk'not in col)
             and ('rate' not in col)
             and ('1st' not in col)
             and ('lst' not in col)):
            aggr_dp_columns.append(col)
dp_features_full  = []
for col in aggr_dp_columns:
    dp_features_full = dp_features_full + [f'{col}_sum_3m', f'{col}_sum_6m', f'{col}_sum_9m', f'{col}_sum_12m', f'{col}_sum_3_6', f'{col}_sum_3_9', 
                                           f'{col}_sum_3_12', f'{col}_sum_6_9', f'{col}_sum_6_12', f'{col}_sum_9_12']
            
querry1 = ''
querry2 = ''
for col in aggr_dp_columns:
    querry1 = querry1 + f'''coalesce({col}, 0) as {col},
                         '''
    querry2 = querry2 + f'''     
                CAST(sum(coalesce(agg_3m.{col},0)) AS double) as {col}_sum_3m,
                CAST(sum(coalesce(agg_6m.{col},0)) AS double) as {col}_sum_6m,
                CAST(sum(coalesce(agg_9m.{col},0)) AS double) as {col}_sum_9m,
                CAST(sum(coalesce(agg_12m.{col},0)) AS double) as {col}_sum_12m,
                CAST(sum(coalesce(agg_3m.{col},0)) / sum(coalesce(agg_6m.{col},0)) AS double) AS {col}_sum_3_6,
                CAST(sum(coalesce(agg_3m.{col},0)) / sum(coalesce(agg_9m.{col},0)) AS double) AS {col}_sum_3_9,
                CAST(sum(coalesce(agg_3m.{col},0)) / sum(coalesce(agg_12m.{col},0)) AS double) AS {col}_sum_3_12,
                CAST(sum(coalesce(agg_6m.{col},0)) / sum(coalesce(agg_9m.{col},0)) AS double) AS {col}_sum_6_9,
                CAST(sum(coalesce(agg_6m.{col},0)) / sum(coalesce(agg_12m.{col},0)) AS double) AS {col}_sum_6_12,
                CAST(sum(coalesce(agg_9m.{col},0)) / sum(coalesce(agg_12m.{col},0)) AS double) AS {col}_sum_9_12,'''
querry1_clean = ''
querry2_clean = ''
for col in set(dp_features_full).intersection(set(model.feature_names_)):
    for i in querry1.split('\n'):
        if ((col[:-8] in i) | (col[:-7] in i)| (col[:-9] in i)) & (col[:-8] not in querry2_clean) & (col[:-7] not in querry2_clean) & (col[:-9] not in querry2_clean) :
            querry1_clean = querry1_clean + i + '\n'
            
    for i in querry2.split('\n'):
        if (col in i):
            querry2_clean = querry2_clean + i + '\n'

In [22]:
base = spark.sql(f'''
SELECT
    '{report_dt}' as report_dt,
    epk_id,
    target,
    last_day(add_months('{report_dt}', -2)) AS report_dt_3m,
    last_day(add_months('{report_dt}', -5)) AS report_dt_6m,
    last_day(add_months('{report_dt}', -8)) AS report_dt_9m,
    last_day(add_months('{report_dt}', -11)) AS report_dt_12m
FROM
    arnsdpsbx_team_ss.bpm_premier_multiclass_new_target_v3_oot
''')





aggr = spark.sql(f'''
SELECT
    epk_id,
    '{report_dt}' as report_dt,
    {agg_features}
FROM 
    {tables_dict['agg']}
WHERE
    report_dt = '{report_dt}'
    and sd_dead_nflag = 0
    and cla_all_active_1m_nflag = 1
''')



agg_deep = spark.sql(f'''
    SELECT
        '{report_dt}' as report_dt,
        epk_id,
        {querry1_clean[:-2]}
    FROM
        {tables_dict['agg']}
    WHERE
        report_dt BETWEEN last_day(add_months('{report_dt}', -11)) AND '{report_dt}'
''')



base.createOrReplaceTempView('base')
aggr.createOrReplaceTempView('aggr')
agg_deep.createOrReplaceTempView('agg_deep')

agg_dynamic_features = spark.sql(f'''
    SELECT
        base.report_dt,
        base.epk_id,
        {querry2_clean[:-2]}
    FROM
        base
        LEFT JOIN agg_deep agg_3m USING (epk_id)
        LEFT JOIN agg_deep agg_6m USING (epk_id)
        LEFT JOIN agg_deep agg_9m USING (epk_id)
        LEFT JOIN agg_deep agg_12m USING (epk_id)
    WHERE
        agg_3m.report_dt BETWEEN base.report_dt_3m AND base.report_dt
        AND agg_6m.report_dt BETWEEN base.report_dt_6m AND base.report_dt
        AND agg_9m.report_dt BETWEEN base.report_dt_9m AND base.report_dt
        AND agg_12m.report_dt BETWEEN base.report_dt_12m AND base.report_dt
    GROUP BY
        base.report_dt,
        base.epk_id
''')

agg_dynamic_features.createOrReplaceTempView('agg_dynamic_features')





all_features = spark.sql(f'''
SELECT DISTINCT
    epk_id,
    report_dt,
    target,
    {features}

FROM
    base
    LEFT JOIN aggr using(report_dt, epk_id)
    LEFT JOIN agg_dynamic_features using(report_dt, epk_id)
''')

all_features_fix = fix_spark_types(all_features)
all_features_fix.write.saveAsTable(f"{output_scheme}.{model_name}_features_scoring_oot_new", mode = 'overwrite')

100%|██████████| 22/22 [00:01<00:00, 21.62it/s]
26/03/19 13:45:22 WARN DAGScheduler: Broadcasting large task binary with size 1441.7 KiB


In [23]:
spark.sql(f'''
select * from {output_scheme}.{model_name}_features_scoring_oot_new
''').write.parquet('hdfs://arnsdpsbx/user/team/team_ss/bpm_premier_multiclass_v4_oot_model_fixed_new', mode='overwrite')

In [19]:
report_dt

'2024-10-31'

In [ ]:
bpm_premier_multiclass_v4_oot_model_fixed - оот на модель просто в4
bpm_premier_multiclass_v4_oot_model_fixed_new - оот на модель в4 новая 